In [1]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd

In [ ]:
DATA       = "data.yaml"
BASE_MODEL = "yolo26n.pt"

# параметры общие для всех обучений
COMMON = dict(
    data          = DATA,
    imgsz         = 640,
    batch         = 16,
    epochs        = 7,
    warmup_epochs = 2,
    device        = 0,
    seed          = 42,
    verbose       = False,
    fliplr=0.0, 
    cls=1.0, 
)

CONFIGS = [
    {
        "name": "1_pretrained",
        # нет обучения — валидируем напрямую
    },
    {
        "name":   "2_scale03",
        "scale":  0.3,
        "box":    10.0,
        "cos_lr": False,
    },
    {
        "name":   "3_scale02",
        "scale":  0.2,
        "box":    10.0,
        "cos_lr": True,
        "hsv_v":  0.5,
    },
    {
        "name":         "4_scale01",
        "scale":        0.1,
        "box":          12.0,
        "cos_lr":       True,
        "hsv_v":        0.5,
        "weight_decay": 3e-4,
    },
]

def get_metrics(val_result) -> dict:
    m = val_result.results_dict
    p = m.get("metrics/precision(B)", 0)
    r = m.get("metrics/recall(B)",    0)
    return {
        "mAP50":     round(m.get("metrics/mAP50(B)",    0), 4),
        "mAP50-95":  round(m.get("metrics/mAP50-95(B)", 0), 4),
        "precision": round(p, 4),
        "recall":    round(r, 4),
        "F1":        round(2 * p * r / (p + r + 1e-9),  4),
    }

In [ ]:

rows = []

for cfg in CONFIGS:
    name  = cfg["name"]
    extra = {k: v for k, v in cfg.items() if k != "name"}
    model = YOLO(BASE_MODEL)

    if not extra:
        val = model.val(data=DATA, imgsz=640, batch=16, device=0, verbose=False)
    else:
        model.train(**COMMON, name=f"exp_{name}", exist_ok=True, **extra)
        
        best_pt = Path(f"runs/detect/exp_{name}/weights/best.pt")
        model   = YOLO(best_pt)
        val     = model.val(data=DATA, imgsz=640, batch=16, device=0, verbose=False)

    row = {"config": name, **get_metrics(val)}
    rows.append(row)
    print(f"[{name}]  mAP50={row['mAP50']}  mAP50-95={row['mAP50-95']}  F1={row['F1']}")


df = (pd.DataFrame(rows)
        .set_index("config")
        .sort_values("mAP50-95", ascending=False))

print("\n" + "=" * 60)
print(df.to_string())
print("=" * 60)
print(f"\nЛучший конфиг: {df['mAP50-95'].idxmax()}")

Ultralytics 8.4.45  Python-3.11.9 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
YOLO26n summary (fused): 122 layers, 2,408,932 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 7.03.2 MB/s, size: 29.3 KB)
val: Scanning C:\Users\pusch\Documents\GitHub\vkr\dataset2\labels\val... 3288 images, 16 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3288/3288 665.9it/s 4.9s0.1s
val: C:\Users\pusch\Documents\GitHub\vkr\dataset2\images\val\Alkan - Posement-001_x0000_y0000.jpg: 13 duplicate labels removed
val: C:\Users\pusch\Documents\GitHub\vkr\dataset2\images\val\Alkan - Posement-001_x0000_y0400.jpg: 6 duplicate labels removed
val: C:\Users\pusch\Documents\GitHub\vkr\dataset2\images\val\Alkan - Posement-001_x0400_y0000.jpg: 2 duplicate labels removed
val: C:\Users\pusch\Documents\GitHub\vkr\dataset2\images\val\Alkan - Posement-001_x0597_y0000.jpg: 2 duplicate labels removed
val: C:\Users\pusch\Documents\GitHub\vkr\dataset2\images\val\Alkan - Posement-

In [3]:
rows=[]

for cfg in CONFIGS:
    name  = cfg["name"]
    extra = {k: v for k, v in cfg.items() if k != "name"}
    model = YOLO(BASE_MODEL)

    if not extra:
        continue
    else:
        best_pt = Path(f"runs/detect/exp_{name}/weights/best.pt")
        model   = YOLO(best_pt)
        test     = model.val(data=DATA, split='test', imgsz=640, batch=16, device=0, verbose=False)

    row = {"config": name, **get_metrics(test)}
    rows.append(row)
    print(f"[{name}]  mAP50={row['mAP50']}  mAP50-95={row['mAP50-95']}  F1={row['F1']}")


df = (pd.DataFrame(rows)
        .set_index("config")
        .sort_values("mAP50-95", ascending=False))

print("\n" + "=" * 60)
print(df.to_string())
print("=" * 60)
print(f"\nЛучший конфиг: {df['mAP50-95'].idxmax()}")

Ultralytics 8.4.45  Python-3.11.9 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
YOLO26n summary (fused): 122 layers, 2,377,956 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 4.60.5 MB/s, size: 28.0 KB)
val: Scanning C:\Users\pusch\Documents\GitHub\vkr\dataset2\labels\test... 3325 images, 19 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3325/3325 605.6it/s 5.5s0.1s
val: C:\Users\pusch\Documents\GitHub\vkr\dataset2\images\test\Alkan - Posement-001_x0000_y0000.jpg: 13 duplicate labels removed
val: C:\Users\pusch\Documents\GitHub\vkr\dataset2\images\test\Alkan - Posement-001_x0000_y0400.jpg: 6 duplicate labels removed
val: C:\Users\pusch\Documents\GitHub\vkr\dataset2\images\test\Alkan - Posement-001_x0400_y0000.jpg: 2 duplicate labels removed
val: C:\Users\pusch\Documents\GitHub\vkr\dataset2\images\test\Alkan - Posement-001_x0597_y0000.jpg: 2 duplicate labels removed
val: C:\Users\pusch\Documents\GitHub\vkr\dataset2\images\test\Alkan - Pos